# Cleansing: เตรียมไฟล์ UNSW-NB15

โค้ดทำความสะอาดอยู่ใน notebook นี้ เพื่อให้เห็นทุกขั้น ไม่เรียก `cleanse_dataset()` จากไฟล์ภายนอก

งานนี้ปรับรูปแบบข้อความ/ชื่อคอลัมน์ แปลง sentinel เป็น missing เพิ่มรหัสแถวและทำเครื่องหมายซ้ำ แล้วบันทึก CSV แบบแบ่ง chunk โดยไม่ลบแถวหรือ clip outlier อัตโนมัติ

> ต้องรัน EDA และตรวจไฟล์ raw ก่อน ไฟล์ต้นฉบับจะไม่ถูกแก้ไข

## 1. เตรียม path และตรวจไฟล์ต้นทาง

รองรับ CSV/CSV.GZ/TXT ใน `raw/` การบันทึก manifest ก่อนและหลังทำงานช่วยยืนยันว่าไฟล์ดิบไม่เปลี่ยนระหว่าง cleansing

In [1]:
from pathlib import Path
import hashlib
import re
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
project_root = next((path for path in (current, *current.parents)
                     if (path / 'data-pipeline' / 'UNSW-NB15' / 'README.md').is_file()), None)
if project_root is None:
    raise FileNotFoundError('เปิด notebook จากภายในโฟลเดอร์โปรเจกต์ CSI_ML_Final')

dataset_dir = project_root / 'data-pipeline' / 'UNSW-NB15'
raw_dir = dataset_dir / 'raw'
output_dir = dataset_dir / 'output' / 'cleansing'
chunk_size = 100_000
supported_suffixes = {'.csv', '.gz', '.txt'}
source_files = sorted(path for path in raw_dir.rglob('*')
                      if path.is_file() and path.suffix.lower() in supported_suffixes)
if not source_files:
    raise FileNotFoundError(f'ไม่พบ CSV ต้นทางใน {raw_dir}; วางไฟล์ UNSW-NB15 ก่อน')

def sha256_file(file_path):
    digest = hashlib.sha256()
    with file_path.open('rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def make_manifest(paths):
    return pd.DataFrame([
        {'file': path.relative_to(raw_dir).as_posix(),
         'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
        for path in paths
    ], columns=['file', 'bytes', 'sha256'])

manifest_before = make_manifest(source_files)
display(manifest_before[['file', 'bytes', 'sha256']])

,file,bytes,sha256
0,UNSW_NB15_testing-set.csv,15380797,aa807a145f06f6883101bb649ce4344b632cc2828abd76...
1,UNSW_NB15_training-set.csv,32293015,5e3eb6037dab1c7eabd79d3358ab89f76c11003979e21c...


## 2. กติกาปรับข้อความและ missing

ตัด BOM/ช่องว่างรอบชื่อคอลัมน์และข้อความ หากชื่อชนกันหลัง normalize จะหยุดให้ตรวจต้นทาง ข้อความว่างและ sentinel ที่สื่อว่าไม่มีค่าให้เป็น missing ไม่ใช่เลขศูนย์

In [2]:
missing_words = {
    '', 'nan', 'null', 'none', 'n/a',
    'inf', '+inf', '-inf', 'infinity', '+infinity', '-infinity',
}

def clean_chunk(raw_chunk):
    cleaned = raw_chunk.copy()
    cleaned.columns = [str(name).replace('\ufeff', '').strip()
                       for name in cleaned.columns]
    if len(set(cleaned.columns)) != len(cleaned.columns):
        raise ValueError('ชื่อคอลัมน์ซ้ำหลังตัด BOM/ช่องว่าง; ตรวจ schema ต้นทาง')

    for column in cleaned.columns:
        text = cleaned[column].astype('string').str.strip()
        is_missing_word = text.str.lower().isin(missing_words)
        cleaned[column] = text.mask(is_missing_word, pd.NA)
    return cleaned

print('จำนวน sentinel ที่กำหนด:', len(missing_words))
print('ตัวอย่าง: ช่องว่างหรือข้อความ n/a จะกลายเป็น missing; ค่า 0 ยังคงเป็น 0')

จำนวน sentinel ที่กำหนด: 11
ตัวอย่าง: ช่องว่างหรือข้อความ n/a จะกลายเป็น missing; ค่า 0 ยังคงเป็น 0


## 3. อ่านทีละ chunk เพิ่มรหัสแถวและทำเครื่องหมาย record ซ้ำ

แต่ละแถวจะได้ `orig_row_id` ที่ประกอบจากชื่อไฟล์และตำแหน่งเดิม ส่วน `_duplicate_flag` บอกว่าเนื้อหาแถวนั้นซ้ำกับแถวก่อนหน้าในไฟล์เดียวกัน เราเก็บแถวซ้ำไว้เพื่อให้ตัดสินใจภายหลังได้จากรายงาน

In [3]:
output_dir.mkdir(parents=True, exist_ok=True)
audit_rows = []

for source_path in source_files:
    relative_name = source_path.relative_to(raw_dir).as_posix()
    stem = source_path.name
    for suffix in ('.gz', '.csv', '.txt'):
        if stem.lower().endswith(suffix):
            stem = stem[:-len(suffix)]
    stem = re.sub(r'[^A-Za-z0-9_.-]+', '_', stem)
    output_path = output_dir / f'{stem}.csv'
    temporary_path = output_dir / f'.{stem}.partial.csv'
    temporary_path.unlink(missing_ok=True)

    read_options = {'dtype': 'string', 'chunksize': chunk_size}
    if source_path.suffix.lower() == '.txt':
        read_options.update({'sep': None, 'engine': 'python'})
    else:
        read_options['low_memory'] = False

    rows_written = 0
    missing_cells = 0
    duplicate_rows = 0
    seen_hashes = set()
    wrote_header = False
    try:
        for chunk in pd.read_csv(source_path, **read_options):
            chunk = clean_chunk(chunk)
            chunk_rows = len(chunk)

            # hash ก่อนเพิ่ม metadata เพื่อเทียบเฉพาะเนื้อหาต้นทางที่ normalize แล้ว
            row_hashes = pd.util.hash_pandas_object(chunk, index=False).astype('uint64')
            duplicate_flags = []
            for row_hash in row_hashes:
                hash_value = int(row_hash)
                duplicate_flags.append(hash_value in seen_hashes)
                seen_hashes.add(hash_value)
            duplicate_rows += sum(duplicate_flags)

            chunk.insert(0, 'orig_row_id',
                         [f'{relative_name}:{row_number:09d}'
                          for row_number in range(rows_written, rows_written + chunk_rows)])
            chunk.insert(1, '_duplicate_flag', duplicate_flags)
            missing_cells += int(chunk.isna().sum().sum())

            # เขียนทีละ chunk และเขียน header เฉพาะครั้งแรก
            chunk.to_csv(
                temporary_path,
                mode='a' if wrote_header else 'w',
                header=not wrote_header,
                index=False,
                encoding='utf-8',
            )
            wrote_header = True
            rows_written += chunk_rows

        if not wrote_header:
            raise ValueError(f'ไม่พบ record ใน {source_path.name}')
        temporary_path.replace(output_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise

    audit_rows.append({
        'file': relative_name, 'rows_in': rows_written, 'rows_out': rows_written,
        'rows_removed': 0, 'duplicate_rows_flagged': duplicate_rows,
        'missing_cells': missing_cells, 'output_file': output_path.name,
    })
    print(f'เสร็จ: {source_path.name} → {output_path.name} ({rows_written:,} แถว)')

เสร็จ: UNSW_NB15_testing-set.csv → UNSW_NB15_testing-set.csv (82,332 แถว)
เสร็จ: UNSW_NB15_training-set.csv → UNSW_NB15_training-set.csv (175,341 แถว)


## 4. ตรวจยอดและบันทึกรายงาน

ตรวจ manifest ซ้ำหลังประมวลผล หาก raw เปลี่ยนระหว่างทางจะไม่ประกาศว่ารอบนี้สำเร็จ รายงาน audit ทำให้เห็นว่าไม่มีแถวถูกลบหรือถูกแทนค่าด้วยศูนย์โดยไม่ตั้งใจ

In [4]:
manifest_after = make_manifest(source_files)
if not manifest_before.equals(manifest_after):
    raise RuntimeError('ไฟล์ raw เปลี่ยนระหว่าง cleansing; ลบผลรอบนี้และรันใหม่จาก snapshot ที่คงที่')

audit_report = pd.DataFrame(audit_rows)
audit_path = output_dir / 'cleansing_audit.csv'
manifest_path = output_dir / 'source_manifest.csv'
audit_report.to_csv(audit_path, index=False)
manifest_after.to_csv(manifest_path, index=False)
display(audit_report)
print('รายงาน audit:', audit_path.relative_to(project_root))
print('manifest ต้นทาง:', manifest_path.relative_to(project_root))
print('จำนวนแถวที่ถูกลบ:', int(audit_report['rows_removed'].sum()))

,file,rows_in,rows_out,rows_removed,duplicate_rows_flagged,missing_cells,output_file
0,UNSW_NB15_testing-set.csv,82332,82332,0,0,0,UNSW_NB15_testing-set.csv
1,UNSW_NB15_training-set.csv,175341,175341,0,0,0,UNSW_NB15_training-set.csv


รายงาน audit: data-pipeline\UNSW-NB15\output\cleansing\cleansing_audit.csv
manifest ต้นทาง: data-pipeline\UNSW-NB15\output\cleansing\source_manifest.csv
จำนวนแถวที่ถูกลบ: 0
